# 00 — Diagnostik: kemiripan gambar test bermasalah vs data train

Notebook **terpisah**, bukan bagian dari alur NB01-04 resmi. Dibuat untuk menjawab satu pertanyaan
spesifik: 38 gambar test yang prediksinya berubah di ensemble percobaan 1 (mayoritas pola
0/1 -> 2, terutama kerajinan kertas berbentuk bunga & kemasan bertema makanan) — apakah data TRAIN
punya cukup contoh visual serupa berlabel benar, sehingga model **bisa** dilatih untuk menangani pola
ini? Atau data train memang tidak punya contoh serupa sama sekali, sehingga ini batasan data (bukan
batasan teknik training)?

**Caranya**: cari tetangga pHash terdekat di data train untuk tiap gambar test bermasalah, lihat
label mayoritas tetangganya.

**Cara membaca hasil di akhir notebook ini:**
- Jarak hamming ke tetangga terdekat **kecil (<15-an)** + label tetangga **konsisten Recyclable**
  -> data train PUNYA sinyal cukup -> pola ini SEHARUSNYA bisa dipelajari, masalahnya ada di resep
  training (augmentasi/durasi), bukan ketiadaan data.
- Jarak hamming **besar (>20-25)** -> tidak ada tetangga mirip di train -> model memang tidak pernah
  lihat pola ini -> batasan data, bukan batasan teknik training (tidak boleh ditambal data luar,
  PRD/aturan panitia).
- Tetangga ada tapi labelnya **campur-campur** -> ada inkonsistensi label untuk pola ini di data
  train sendiri -> bukan sesuatu yang bisa diperbaiki dari sisi training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import imagehash
import pandas as pd
from PIL import Image
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
TRAIN_PROCESSED_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "processed"
FOLD_ASSIGNMENT_PATH = DRIVE_ROOT / "Preprocessing_Output" / "manifests" / "fold_assignment.csv"
TEST_ROOT = DRIVE_ROOT / "BDC2026" / "test"

for _p in [TRAIN_PROCESSED_ROOT, FOLD_ASSIGNMENT_PATH, TEST_ROOT]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} tidak ketemu -- cek Drive ter-mount dan path benar.")

label_names = {0: "Recyclable", 1: "Electronic", 2: "Organic"}


## Hitung pHash seluruh gambar train (sekali saja)

Dijalankan sekali di awal sesi; sisa notebook cuma query terhadap hash yang sudah dihitung.


In [ ]:
fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
label_by_id = fold_assignment.set_index("image_id")["label"]

print(f"Menghitung pHash untuk {len(fold_assignment)} gambar train...")
train_hashes = {}
for row in fold_assignment.itertuples():
    p = TRAIN_PROCESSED_ROOT / f"{row.image_id}.jpg"
    train_hashes[row.image_id] = imagehash.phash(Image.open(p).convert("RGB"), hash_size=8)
print(f"Selesai: {len(train_hashes)} hash tersimpan.")


## 38 id test yang prediksinya berubah di percobaan 1 (0/1 -> 2)

Daftar ini diambil dari perbandingan `submission_KepepetDeadline.csv` (lama) vs hasil ensemble
14-model percobaan 1.


In [ ]:
HARD_TEST_IDS = [6, 14, 19, 60, 116, 132, 293, 312, 363, 372, 413, 438, 499, 508, 565, 637, 659,
                  728, 781, 797, 843, 899, 907, 908, 1033, 1035, 1044, 1079, 1092, 1105, 1136,
                  1141, 1144, 1168, 1199, 1351, 1373, 1440]
print(f"{len(HARD_TEST_IDS)} id test akan dicek.")


## Cari K tetangga pHash terdekat di train untuk tiap gambar test bermasalah


In [ ]:
K = 5  # jumlah tetangga train terdekat yang dicek per gambar test

results = []
for test_id in HARD_TEST_IDS:
    test_hash = imagehash.phash(Image.open(TEST_ROOT / f"{test_id}.jpg").convert("RGB"), hash_size=8)
    distances = sorted(((tid, test_hash - th) for tid, th in train_hashes.items()), key=lambda x: x[1])
    nearest = distances[:K]
    labels_nearest = [label_by_id.loc[tid] for tid, _ in nearest]
    label_counts = pd.Series(labels_nearest).map(label_names).value_counts().to_dict()
    dists = [d for _, d in nearest]
    results.append({
        "test_id": test_id,
        "min_hamming_dist": min(dists),
        "hamming_dists": dists,
        "label_tetangga": label_counts,
        "ada_tetangga_mirip": min(dists) <= 20,
    })

results_df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print(results_df.to_string(index=False))


## Ringkasan kesimpulan


In [ ]:
n_ada_sinyal = results_df["ada_tetangga_mirip"].sum()
n_tanpa_sinyal = len(results_df) - n_ada_sinyal
print(f"Gambar test dengan tetangga train mirip (jarak <=20): {n_ada_sinyal}/{len(results_df)}")
print(f"Gambar test TANPA tetangga train mirip (jarak >20):   {n_tanpa_sinyal}/{len(results_df)}")

print()
print("Detail yang PUNYA tetangga mirip -- cek konsistensi labelnya:")
for r in results:
    if r["ada_tetangga_mirip"]:
        konsisten = len(r["label_tetangga"]) == 1
        status = "KONSISTEN" if konsisten else "CAMPUR/INKONSISTEN"
        print(f"  id={r['test_id']}: {r['label_tetangga']} -> {status}")


## Interpretasi

Isi bagian ini setelah melihat hasil run di atas:
- Kalau mayoritas `ada_tetangga_mirip=True` DAN labelnya konsisten Recyclable -> ada peluang nyata
  memperbaiki lewat augmentasi/training lebih baik (bukan kurang data).
- Kalau mayoritas `ada_tetangga_mirip=False` -> ini batasan data yang tidak bisa ditambal tanpa
  data luar (dilarang aturan kompetisi) -> terima sebagai batas bawah error yang wajar.
